In [ ]:
# ============================================
# 1. ОБЩАЯ ЧАСТЬ ДЛЯ ВСЕХ ТРЁХ МОДЕЛЕЙ
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

# Параметры
BATCH_SIZE = 64
NUM_EPOCHS = 20
IMG_SIZE_RESNET = 224
IMG_SIZE_EFFICIENTNET = 300

# Загрузка данных
train_df = pd.read_csv("Digital_core_v5.2/train_pairs.csv")
val_df = pd.read_csv("Digital_core_v5.2/val_pairs.csv")
classes = sorted(train_df['mineral_grouped'].unique())
class_to_idx = {cls: i for i, cls in enumerate(classes)}
num_classes = len(classes)

# Трансформации
transform_resnet = transforms.Compose([
    transforms.Resize((IMG_SIZE_RESNET, IMG_SIZE_RESNET)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_efficientnet = transforms.Compose([
    transforms.Resize((IMG_SIZE_EFFICIENTNET, IMG_SIZE_EFFICIENTNET)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Датасет
class SiamesePairDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.labels = [class_to_idx[cls] for cls in self.df['mineral_grouped']]
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ds_img = Image.open(row['ds_path']).convert('RGB')
        uv_img = Image.open(row['uv_path']).convert('RGB')
        if self.transform:
            ds_img = self.transform(ds_img)
            uv_img = self.transform(uv_img)
        return ds_img, uv_img, self.labels[idx]

# Sampler
def get_sampler(dataset):
    class_counts = np.bincount(dataset.labels)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[label] for label in dataset.labels]
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# Функции обучения и оценки
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    pbar = tqdm(loader, desc="Train")
    for ds, uv, labels in pbar:
        ds, uv, labels = ds.to(device), uv.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(ds, uv)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = torch.max(out, 1)
        correct += (pred == labels).sum().item()
        total += len(labels)
        pbar.set_postfix({"Loss": f"{loss.item():.3f}", "Acc": f"{100*correct/total:.1f}"})
    return total_loss / len(loader), 100 * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for ds, uv, labels in tqdm(loader, desc="Val"):
            ds, uv, labels = ds.to(device), uv.to(device), labels.to(device)
            out = model(ds, uv)
            loss = criterion(out, labels)
            total_loss += loss.item()
            _, pred = torch.max(out, 1)
            correct += (pred == labels).sum().item()
            total += len(labels)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), 100 * correct / total, all_preds, all_labels

def plot_metrics(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14,5))
    axes[0].plot(history['train_acc'], label='Train Acc')
    axes[0].plot(history['val_acc'], label='Val Acc')
    axes[0].set_title(f"{model_name} – Accuracy")
    axes[0].legend()
    axes[0].grid()
    axes[1].plot(history['train_loss'], label='Train Loss')
    axes[1].plot(history['val_loss'], label='Val Loss')
    axes[1].set_title(f"{model_name} – Loss")
    axes[1].legend()
    axes[1].grid()
    plt.savefig(f"{model_name}_metrics.png")
    plt.show()

def plot_confusion_matrix(all_labels, all_preds, model_name):
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f"{model_name} – Confusion Matrix")
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f"{model_name}_cm.png")
    plt.show()

In [ ]:
# ============================================
# 2. МОДЕЛЬ 1: RESNET18
# ============================================

class SiameseResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = self.encoder.fc.in_features
        self.encoder.fc = nn.Identity()
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.classifier = nn.Linear(in_features * 2, num_classes)
    
    def forward(self, x1, x2):
        f1 = self.encoder(x1)
        f2 = self.encoder(x2)
        return self.classifier(torch.cat((f1, f2), dim=1))

# Датасеты
train_ds = SiamesePairDataset(train_df, transform=transform_resnet)
val_ds = SiamesePairDataset(val_df, transform=transform_resnet)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=get_sampler(train_ds))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

model = SiameseResNet(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), "best_siamese_resnet.pth")
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%, Best={best_val:.2f}%")

# Финальная оценка
_, _, all_preds, all_labels = evaluate(model, val_loader, criterion)
print("\nClassification Report (ResNet18):")
print(classification_report(all_labels, all_preds, target_names=classes))
plot_metrics(history, "ResNet18_Siamese")
plot_confusion_matrix(all_labels, all_preds, "ResNet18_Siamese")

In [ ]:
# ============================================
# 3. МОДЕЛЬ 2: MOBILENETV3 (исправленная, с динамическим определением in_features)
# ============================================

class SiameseMobileNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        self.encoder.classifier = nn.Identity()
        # Не замораживаем и не определяем in_features здесь
        # Сделаем позже
        self.num_classes = num_classes
        self.in_features = None
        self.classifier = None
    
    def build_classifier(self, device):
        # Переносим encoder на нужное устройство
        self.encoder = self.encoder.to(device)
        # Замораживаем
        for param in self.encoder.parameters():
            param.requires_grad = False
        # Определяем in_features
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224).to(device)
            features = self.encoder(dummy)
            self.in_features = features.shape[1]
        print(f"MobileNet in_features: {self.in_features}")
        self.classifier = nn.Linear(self.in_features * 2, self.num_classes).to(device)
    
    def forward(self, x1, x2):
        f1 = self.encoder(x1)
        f2 = self.encoder(x2)
        return self.classifier(torch.cat((f1, f2), dim=1))

# Датасеты (используем transform_resnet, так как MobileNet тоже 224x224)
train_ds_mobilenet = SiamesePairDataset(train_df, transform=transform_resnet)
val_ds_mobilenet = SiamesePairDataset(val_df, transform=transform_resnet)

train_loader_mobilenet = DataLoader(train_ds_mobilenet, batch_size=BATCH_SIZE, sampler=get_sampler(train_ds_mobilenet))
val_loader_mobilenet = DataLoader(val_ds_mobilenet, batch_size=BATCH_SIZE, shuffle=False)

model_mobilenet = SiameseMobileNet(num_classes)
model_mobilenet.build_classifier(device)
model_mobilenet = model_mobilenet.to(device)
optimizer_mobilenet = optim.Adam(model_mobilenet.classifier.parameters(), lr=0.001)
scheduler_mobilenet = optim.lr_scheduler.ReduceLROnPlateau(optimizer_mobilenet, mode='min', factor=0.5, patience=3)

history_mobilenet = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_mobilenet = 0
criterion = nn.CrossEntropyLoss()
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model_mobilenet, train_loader_mobilenet, criterion, optimizer_mobilenet)
    val_loss, val_acc, _, _ = evaluate(model_mobilenet, val_loader_mobilenet, criterion)
    scheduler_mobilenet.step(val_loss)
    history_mobilenet['train_loss'].append(train_loss)
    history_mobilenet['train_acc'].append(train_acc)
    history_mobilenet['val_loss'].append(val_loss)
    history_mobilenet['val_acc'].append(val_acc)
    if val_acc > best_val_mobilenet:
        best_val_mobilenet = val_acc
        torch.save(model_mobilenet.state_dict(), "best_siamese_mobilenet.pth")
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%, Best={best_val_mobilenet:.2f}%")

_, _, all_preds_mobilenet, all_labels_mobilenet = evaluate(model_mobilenet, val_loader_mobilenet, criterion)
print("\nClassification Report (MobileNet):")
print(classification_report(all_labels_mobilenet, all_preds_mobilenet, target_names=classes))
plot_metrics(history_mobilenet, "MobileNet_Siamese")
plot_confusion_matrix(all_labels_mobilenet, all_preds_mobilenet, "MobileNet_Siamese")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

# Загрузка данных (пути могут отличаться – проверьте)
train_df = pd.read_csv("Digital_core_v5.2/train_pairs.csv")
val_df = pd.read_csv("Digital_core_v5.2/val_pairs.csv")
classes = sorted(train_df['mineral_grouped'].unique())
class_to_idx = {cls: i for i, cls in enumerate(classes)}
num_classes = len(classes)

# Трансформации для EfficientNet
transform_efficientnet = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Датасет
class SiamesePairDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.labels = [class_to_idx[cls] for cls in self.df['mineral_grouped']]
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ds_img = Image.open(row['ds_path']).convert('RGB')
        uv_img = Image.open(row['uv_path']).convert('RGB')
        if self.transform:
            ds_img = self.transform(ds_img)
            uv_img = self.transform(uv_img)
        return ds_img, uv_img, self.labels[idx]

# Sampler
def get_sampler(dataset):
    class_counts = np.bincount(dataset.labels)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[label] for label in dataset.labels]
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# Функции обучения и оценки
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    pbar = tqdm(loader, desc="Train")
    for ds, uv, labels in pbar:
        ds, uv, labels = ds.to(device), uv.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(ds, uv)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = torch.max(out, 1)
        correct += (pred == labels).sum().item()
        total += len(labels)
        pbar.set_postfix({"Loss": f"{loss.item():.3f}", "Acc": f"{100*correct/total:.1f}"})
    return total_loss / len(loader), 100 * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for ds, uv, labels in tqdm(loader, desc="Val"):
            ds, uv, labels = ds.to(device), uv.to(device), labels.to(device)
            out = model(ds, uv)
            loss = criterion(out, labels)
            total_loss += loss.item()
            _, pred = torch.max(out, 1)
            correct += (pred == labels).sum().item()
            total += len(labels)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), 100 * correct / total, all_preds, all_labels

def plot_metrics(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14,5))
    axes[0].plot(history['train_acc'], label='Train Acc')
    axes[0].plot(history['val_acc'], label='Val Acc')
    axes[0].set_title(f"{model_name} – Accuracy")
    axes[0].legend()
    axes[0].grid()
    axes[1].plot(history['train_loss'], label='Train Loss')
    axes[1].plot(history['val_loss'], label='Val Loss')
    axes[1].set_title(f"{model_name} – Loss")
    axes[1].legend()
    axes[1].grid()
    plt.savefig(f"{model_name}_metrics.png")
    plt.show()

def plot_confusion_matrix(all_labels, all_preds, model_name):
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f"{model_name} – Confusion Matrix")
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f"{model_name}_cm.png")
    plt.show()

# Модель Siamese с EfficientNet
class SiameseEfficientNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        self.encoder.classifier = nn.Identity()
        self.num_classes = num_classes
        self.classifier = None
    
    def build_classifier(self, device):
        self.encoder = self.encoder.to(device)
        for param in self.encoder.parameters():
            param.requires_grad = False
        with torch.no_grad():
            dummy = torch.randn(1, 3, 300, 300).to(device)
            features = self.encoder(dummy)
            in_features = features.shape[1]
        print(f"EfficientNet in_features: {in_features}")
        self.classifier = nn.Linear(in_features * 2, self.num_classes).to(device)
    
    def forward(self, x1, x2):
        f1 = self.encoder(x1)
        f2 = self.encoder(x2)
        return self.classifier(torch.cat((f1, f2), dim=1))

# Параметры
BATCH_SIZE = 32   # уменьшите до 16, если не хватает памяти
NUM_EPOCHS = 15
LEARNING_RATE = 0.001

train_ds = SiamesePairDataset(train_df, transform=transform_efficientnet)
val_ds = SiamesePairDataset(val_df, transform=transform_efficientnet)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=get_sampler(train_ds))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

model = SiameseEfficientNet(num_classes)
model.build_classifier(device)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), "best_siamese_efficientnet.pth")
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%, Best={best_val:.2f}%")

_, _, all_preds, all_labels = evaluate(model, val_loader, criterion)
print("\nClassification Report (EfficientNet):")
print(classification_report(all_labels, all_preds, target_names=classes))
plot_metrics(history, "EfficientNet_Siamese")
plot_confusion_matrix(all_labels, all_preds, "EfficientNet_Siamese")